In [1]:
#create ipynb file in github

In [2]:
# Import libraries
import os
import pandas as pd
from pathlib import Path

In [ ]:
from pathlib import Path

DATA_PATH = Path("synthetic_financial_data.csv")
df = pd.read_csv(DATA_PATH)
df.head()


,CustomerID,Age,Education,EmploymentType,EmploymentSector,EmploymentLengthYears,MonthlyIncome,MaritalStatus,Dependents,PropertyOwnership,...,MonthlyDebt,YearsWithBank,HasSavingsAccount,HasCheckingAccount,LoanPurpose,LoanAmount,LoanTermMonths,InterestRate,LoanDefault,LoanPurposeDescription
0,1,59,High School,Self-Employed,Technology,11,9067.159786,Single,2,Own with Mortgage,...,1583.065500,7,1,0,Personal,20789.005502,36,3.828919,0,Need funds for the next three years.
1,2,51,Below High School,Full-Time,Government,24,5068.058742,Married,1,Own with Mortgage,...,1332.052087,7,1,1,Personal,27461.583712,48,8.855769,1,I need a personal loan for this to be done.
2,3,24,Bachelor,Unemployed,Manufacturing,0,2557.541824,Single,0,Living with Parents,...,911.536097,2,1,1,Business,68862.744645,240,14.748993,1,Seeking financing to start my own company.
3,4,25,Bachelor,Full-Time,Finance,0,7535.097299,Single,0,Living with Parents,...,514.141526,6,1,0,Wedding,8841.450954,48,4.825886,0,I need a personal loan for me.
4,5,25,Master,Part-Time,Government,5,4798.465251,Married,2,Rent,...,2386.488285,3,0,1,Education,17787.927594,120,18.384157,1,"Applying for student financing for tuition, fe..."


In [4]:
df['LoanPurposeDescription'] = df['LoanPurposeDescription'].fillna("no description")

Sentiment analysis was used to quantify emotional tone in loan purpose descriptions, which may reflect financial risk behavior.

In [5]:
from textblob import TextBlob

df['sentiment_score'] = df['LoanPurposeDescription'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)

Turning important words in text into numerical features that help the model detect financial risk.

In [6]:
risk_keywords = ['debt', 'urgent', 'medical', 'late', 'repay', 'loss']

df['risk_keyword_count'] = df['LoanPurposeDescription'].apply(
    lambda x: sum(word in str(x).lower() for word in risk_keywords)
)

TF-IDF Feature Extraction

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=8,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['LoanPurposeDescription'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

df = pd.concat([df.reset_index(drop=True), tfidf_df], axis=1)

Encode Categorical Variables

In [8]:
df_encoded = pd.get_dummies(
    df,
    columns=[
        'Education', 'EmploymentType', 'EmploymentSector',
        'MaritalStatus', 'PropertyOwnership', 'LoanPurpose'
    ],
    drop_first=True
)


Feature Selection (Clean & Safe)

In [9]:
X = df_encoded.drop(
    columns=['CustomerID', 'LoanDefault', 'LoanPurposeDescription']
)
y = df_encoded['LoanDefault']


Logistic Regression Model

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.755
              precision    recall  f1-score   support

           0       0.74      0.73      0.73        92
           1       0.77      0.78      0.77       108

    accuracy                           0.76       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.75      0.76      0.75       200



Random Forest as Second Model to do comparison with Logistic Regression model.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.76
